In [ ]:
import pandas as pd
from pathlib import Path

_current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in (_current_dir, *_current_dir.parents)
    if (path / "notebooks").is_dir() and (path / "data").is_dir()
)
RESULTS_DIR = PROJECT_ROOT / "results"


## SOL 주요 지갑 추출

일별 PageRank와 betweenness centrality 상위 15개에 포함된 지갑을 중복 제거해 저장합니다.


In [ ]:
def extract_top_wallets(metrics, top_n=15):
    top_pagerank = (
        metrics.sort_values(["date", "pagerank"], ascending=[True, False])
        .groupby("date")
        .head(top_n)
    )
    top_betweenness = (
        metrics.sort_values(["date", "betweenness_centrality"], ascending=[True, False])
        .groupby("date")
        .head(top_n)
    )

    address_column = "address" if "address" in metrics.columns else "Unnamed: 0"
    addresses = set(top_pagerank[address_column]) | set(top_betweenness[address_column])
    return pd.DataFrame({"address": sorted(addresses)})


In [ ]:
metrics_dir = RESULTS_DIR / "solana" / "network_metrics"
output_dir = RESULTS_DIR / "solana" / "wallet_analysis"
output_dir.mkdir(parents=True, exist_ok=True)

metric_groups = {
    "*_centrality.csv": "sol_top_wallets_all_periods.csv",
    "*_centrality_weight.csv": "sol_weighted_top_wallets.csv",
}

for pattern, output_name in metric_groups.items():
    metrics_paths = sorted(metrics_dir.glob(pattern))
    if not metrics_paths:
        print(f"No files matching {pattern} in {metrics_dir}")
        continue

    all_metrics = pd.concat((pd.read_csv(path) for path in metrics_paths), ignore_index=True)
    extract_top_wallets(all_metrics).to_csv(output_dir / output_name, index=False)


In [ ]:
historical_path = metrics_dir / "SOL_2020-03-26_to_2020-12-31_centrality.csv"
if historical_path.exists():
    historical_metrics = pd.read_csv(historical_path)
    historical_top_wallets = extract_top_wallets(historical_metrics)
    historical_top_wallets.to_csv(output_dir / "sol_2020_top_wallets.csv", index=False)
else:
    print(f"Historical metrics file not found: {historical_path}")
